# Saluki for synonymous eGFP CDS design

[Notebook](saluki-egfp.ipynb) · [Installation](../docs/installation.md)

Score a single-exon eGFP transcript with fixed UTRs. Optimize the CDS with synonymous substitutions and retain the terminal stop codon. Install MEME Suite with the notebook dependencies. The first model load downloads the pretrained checkpoint.

## 1. Load the model



In [1]:
from pathlib import Path
from tempfile import mkdtemp

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
cds = "".join(Path("data/egfp.fasta").read_text().splitlines()[1:])
utr5 = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"
utr3 = "GCTGGAGCCTCGGTGGCCTAGCTTCTTGCCCCTTGGGCCTCCCCCCAGCCCCTCCTCCCCTTCCTGCACCCGTACCCCCGTGGTCTTTGAATAAAGTCTGAGTGGGCGGCA"
saluki = load_model(
    "saluki", device=device,
    cds_start=len(utr5), cds_end=len(utr5) + len(cds),
    exon_lengths=[len(utr5) + len(cds) + len(utr3)],
)


def encode_utr(sequence):
    indices = torch.tensor(["ACGT".index(base) for base in sequence])
    return torch.nn.functional.one_hot(indices, num_classes=4)


left, right = encode_utr(utr5), encode_utr(utr3)


class CDSScorer:
    task_names = saluki.task_names
    device = saluki.device

    def score(self, sequences):
        return saluki.score([utr5 + sequence + utr3 for sequence in sequences])

    def score_soft(self, probabilities):
        batch = len(probabilities)
        full_transcripts = torch.cat([
            left.to(probabilities).expand(batch, -1, -1),
            probabilities,
            right.to(probabilities).expand(batch, -1, -1),
        ], dim=1)
        return saluki.score_soft(full_transcripts)


model = CDSScorer()
seeds = [cds]
print("Starting Saluki score:", float(model.score(seeds)[0, 0]))

run = Path(mkdtemp(prefix="motifsift-example-"))


Starting Saluki score: 1.0143448114395142




## 2. Generate candidates

Run the eleven optimizers and keep 100 candidates per optimizer. The saved design summary comes from the previous full search. The motif and panel tables come from reanalyzing its 90% gain candidates.



In [2]:
designs = generate_designs(
    model, length=len(cds), initial=seeds, objective="maximize",
    gain_fractions=(0.5, 0.9), output_dir=run / "designs",
    is_CDS_optimized=True, fixed_region=[slice(len(cds) - 3, len(cds))],
)
designs.groupby(["algorithm", "gain_fraction"])["score"].agg(["size", "median", "max"])


size    median       max
algorithm                       gain_fraction                          
adalead                         0.5             100  1.516985  1.681509
                                0.9             100  2.588798  2.645017
basesearch                      0.5             100  1.433591  1.508610
                                0.9             100  2.516793  2.552695
beam_search                     0.5             100  1.562780  1.689251
                                0.9             100  2.678402  2.724082
cross_entropy_method            0.5             100  1.453089  1.590080
                                0.9             100  2.483534  2.522873
den                             0.5             100  1.396423  1.745093
                                0.9             100  2.107789  2.196756
fast_seqprop                    0.5             100  1.283291  1.754693
                                0.9             100  2.252657  2.447331
genetic_algorithm               0.5             100  1.565318  1.828779
                                0.9             100  2.651065  2.722817
ism_greedy                      0.5             100  1.379088  1.530929
                                0.9             100  2.370621  2.487951
ledidi                          0.5             100  0.787613  1.078837
                                0.9             100  1.319844  1.483486
simulated_annealing             0.5             100  0.591725  1.109233
                                0.9             100  0.913594  1.109233
vanilla_activation_maximization 0.5             100  1.501908  1.996902
                                0.9             100  2.235355  2.428951



## 3. Discover motifs

Analyze the 90% gain stage with synonymous-codon attribution and forward-strand motif discovery.



In [3]:
designs = designs.loc[designs["gain_fraction"].eq(0.9)].copy()
motifs = call_motifs(
    designs, model, reverse_complement=False,
    output_dir=run / "motifs",
)
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
motif_id,,,,
2-ACGCUUAAGU,2-ACGCUUAAGU,635,635,1095.0
3-GUACCACUGGM,3-GUACCACUGGM,475,475,973.0
7-CUACGCUC,7-CUACGCUC,23,23,214.0
8-ACAAUUACAAU,8-ACAAUUACAAU,10,10,2538.0
9-UUAAUGGG,9-UUAAUGGG,17,17,165.0
14-GCAGCGUGCA,14-GCAGCGUGCA,12,12,601.0
22-UUAGCGGCGAG,22-UUAGCGGCGAG,63,63,882.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,ATGGTTAGTAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCC...,2.724082,2-ACGCUUAAGU;3-GUACCACUGGM;8-ACAAUUACAAU;22-UU...
1,2,ATGGTTTCGAAGGGGGAGGAGCTGTTTACGGGGGTGGTGCCGATCT...,2.250725,8-ACAAUUACAAU
2,3,ATGGTGAGCAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCGATCT...,2.295541,8-ACAAUUACAAU;14-GCAGCGUGCA;22-UUAGCGGCGAG
3,4,ATGGTGAGTAAGGGGGAGGAGTTGTTTACGGGGGTGGTTCCGATCT...,2.265943,2-ACGCUUAAGU;8-ACAAUUACAAU
4,5,ATGGTTAGCAAGGGGGAGGAGTTGTTTACGGGGGTCGTGCCCATCC...,2.235750,3-GUACCACUGGM;8-ACAAUUACAAU
5,6,ATGGTTTCGAAGGGGGAAGAGTTGTTCACTGGGGTCGTTCCGATCC...,2.502169,2-ACGCUUAAGU;3-GUACCACUGGM;7-CUACGCUC;8-ACAAUU...
6,7,ATGGTTTCGAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCTATCC...,2.276869,2-ACGCUUAAGU;8-ACAAUUACAAU;14-GCAGCGUGCA
7,8,ATGGTTAGCAAGGGGGAGGAGTTGTTTACGGGGGTGGTGCCGATCC...,2.110986,3-GUACCACUGGM;7-CUACGCUC;8-ACAAUUACAAU;14-GCAG...
8,9,ATGGTTTCGAAGGGGGAGGAGTTGTTCACTGGGGTGGTGCCGATCC...,2.692230,2-ACGCUUAAGU;3-GUACCACUGGM;8-ACAAUUACAAU;9-UUA...
9,10,ATGGTTAGCAAGGGGGAGGAGTTGTTTACGGGGGTCGTGCCCATCC...,2.260627,3-GUACCACUGGM;8-ACAAUUACAAU;22-UUAGCGGCGAG
